In [ ]:
from sage.all import graphs, table, show
from sage.env import SAGE_VERSION

K4 = graphs.CompleteGraph(4)
K5 = graphs.CompleteGraph(5)
K6_minus = graphs.CompleteGraph(6)
K6_minus.delete_edges([(0, 1), (2, 3), (4, 5)])

def forest_count(G):
    # Forests may contain isolated vertices.
    return int(G.tutte_polynomial()(2, 1))

# Check several known spanning-forest counts.
for G, value in [(graphs.CycleGraph(3), 7), (K4, 38),
                 (K5, 291), (K6_minus, 1083)]:
    if forest_count(G) != value:
        raise RuntimeError("Incorrect spanning-forest count")

print("SageMath {}".format(SAGE_VERSION), flush=True)
print(
    "Connected simple graphs up to isomorphism, "
    "3 <= n <= 9, degrees in {2,3,4}.",
    flush=True
)

rows = []

for n in range(3, 10):
    count = tests1 = tests2 = failures1 = failures2 = 0

    for G in graphs.nauty_geng("{} -c -d2 -D4".format(n)):
        count += 1
        degrees = G.degree()
        n2, n3, n4 = [degrees.count(d) for d in (2, 3, 4)]
        F = forest_count(G)

        # Theorem 1.1: degrees in {2,3}, excluding K4.
        if n4 == 0 and not (n == 4 and G.is_isomorphic(K4)):
            tests1 += 1

            # Compare fourth powers to avoid fractional exponents.
            if F**4 < 2**(4*(n2 + n3 - 1)) * 3**(n3 + 2):
                failures1 += 1
                print("Counterexample to Theorem 1.1:", G.graph6_string(),
                      "F =", F, "(n2, n3, n4) =", (n2, n3, n4))

        # Theorem 1.2: exclude K5 and K6 minus a perfect matching.
        exceptional = (
            (n == 5 and G.is_isomorphic(K5)) or
            (n == 6 and G.is_isomorphic(K6_minus))
        )

        if not exceptional:
            tests2 += 1
            # Compare tenth powers and use 198 = 2*99.
            a = 10*n2 + 7*n3 + 4*n4 - 16
            b = n3 + 2*n4 + 2

            if F**10 < 2**a * 99**b:
                failures2 += 1
                print("Counterexample to Theorem 1.2:", G.graph6_string(),
                      "F =", F, "(n2, n3, n4) =", (n2, n3, n4))

    row = (n, count, tests1, tests2, failures1, failures2)
    rows.append(row)
    print("n = {}: {} graphs.".format(n, count), flush=True)

headers = [
    "Vertices", "Graphs",
    "Theorem 1.1 checks", "Theorem 1.2 checks",
    "Theorem 1.1 failures", "Theorem 1.2 failures"
]

totals = ("Total",) + tuple(
    sum(row[j] for row in rows) for j in range(1, 6)
)

show(table(
    rows + [totals],
    header_row=headers,
    frame=True,
    align="center"
))

if any(row[4] or row[5] for row in rows):
    raise RuntimeError("A counterexample was found")

print("All checks passed.", flush=True)
